In [5]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import MinMaxScaler
from nltk.util import ngrams
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Download necessary NLTK resources
nltk.download('punkt')
nltk.download('wordnet')

# Load data
df = pd.read_csv("C:/Users/Acer/Desktop/news_output.csv")

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Preprocessing function
def preprocess_text(text):
    tokens = word_tokenize(str(text).lower())
    return ' '.join([lemmatizer.lemmatize(t) for t in tokens if t.isalnum()])

# Function to compute Jaccard Similarity (with fallback to bigrams or unigrams)
def jaccard_similarity(text1, text2):
    tokens1 = list(ngrams(word_tokenize(str(text1).lower()), 3))  # Convert to trigrams
    tokens2 = list(ngrams(word_tokenize(str(text2).lower()), 3))

    # If not enough trigrams, fall back to bigrams or unigrams
    if len(tokens1) == 0 or len(tokens2) == 0:
        tokens1 = list(ngrams(word_tokenize(str(text1).lower()), 2))  # Bigrams
        tokens2 = list(ngrams(word_tokenize(str(text2).lower()), 2))
    
    if len(tokens1) == 0 or len(tokens2) == 0:
        tokens1 = list(ngrams(word_tokenize(str(text1).lower()), 1))  # Unigrams
        tokens2 = list(ngrams(word_tokenize(str(text2).lower()), 1))

    # Convert to sets and calculate Jaccard similarity
    set_tokens1 = set(tokens1)
    set_tokens2 = set(tokens2)
    
    intersection = len(set_tokens1.intersection(set_tokens2))
    union = len(set_tokens1.union(set_tokens2))
    
    # Return Jaccard similarity, avoiding division by zero
    return intersection / union if union != 0 else 0

# Function to compute TF-IDF Cosine Similarity (as a fallback)
def cosine_similarity_fallback(text1, text2):
    vectorizer = TfidfVectorizer()
    vectors = vectorizer.fit_transform([text1, text2])
    return cosine_similarity(vectors[0], vectors[1])[0][0]

# Preprocess the text columns
df['processed_headline'] = df['headline'].apply(preprocess_text)
df['processed_short_description'] = df['short_description'].apply(preprocess_text)

# Compute coherence score using Jaccard Similarity (with fallback) + Cosine Similarity
coherence_scores = []
for _, row in df.iterrows():
    headline = row['processed_headline']
    short_description = row['processed_short_description']
    
    # Handle short texts: ensure both headline and short_description are long enough
    if len(headline.split()) < 3 or len(short_description.split()) < 3:
        coherence_score = 0.1  # Assign a small baseline score for short texts
    else:
        # Jaccard Similarity (with fallback to bigrams/unigrams)
        score_jaccard = jaccard_similarity(headline, short_description)
        
        # If Jaccard Similarity is 0, use Cosine Similarity (TF-IDF as a backup)
        if score_jaccard == 0:
            score_cosine = cosine_similarity_fallback(headline, short_description)
        else:
            score_cosine = score_jaccard
        
        # Combine both scores (Jaccard + Cosine) for better overall accuracy
        coherence_score = (score_jaccard * 0.6) + (score_cosine * 0.4)

    # Add the coherence score to the list
    coherence_scores.append(coherence_score)

# Add coherence scores to DataFrame
df['coherence_score'] = coherence_scores

# Normalize coherence scores between 0 and 1
scaler = MinMaxScaler()
df['coherence_score'] = scaler.fit_transform(df[['coherence_score']])

# Save the updated DataFrame
df.to_csv("C:/Users/Acer/Desktop/news_articles_with_improved_coherence.csv", index=False)

print("Coherence scores calculated and saved successfully with improved handling!")


Coherence scores calculated and saved successfully with improved handling!


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
